# BƯỚC 2 · TRÍCH ĐẶC TRƯNG & NHẬN DẠNG
**Nhóm 05 · Nhận dạng mẫu · Đề tài 10 Voice Cloning**

Notebook này dùng lại output đã phục hồi, không chạy lại bộ sinh âm thanh. Nó loại **15 clip ghép cặp** tương ứng 5 câu đã xác nhận lỗi, resample về 16 kHz, trích đặc trưng, huấn luyện SVM/Random Forest cho **T-B (bộ sinh)** và **T-C (nhãn giọng)**, tạo HTML và ZIP.

### Trước khi chạy — không cần sửa code
1. Kaggle → **Add Input** → notebook `duyninhus/phuhoi-sinhbu-tudong`, output chứa thư mục `pa2`.
2. Settings: **CPU** là đủ. Internet chỉ cần khi thư viện Python thiếu phải cài.
3. **Save Version → Save & Run All**. Notebook không cần Input 7 `.pt`, trọng số OmniVoice, VieNeu hoặc OmDiOm-ft.

**Thiết kế:** bộ P là âm tổng hợp, dùng cho T-B/T-C; không dùng ở T-A thật/giả. VieNeu được điều kiện hóa bởi tham chiếu WAV tổng hợp từ prompt, không phải cùng bản thu người thật. Đánh giá theo ID câu tách biệt train/test; các câu có khuôn chữ tương tự vẫn có thể rò rỉ nội dung. T-C đánh giá 7 nhãn prompt, không chứng minh nhận dạng người thật chưa thấy.

Test là tập giữ riêng: SVM đầy đủ là mô hình chính **định trước**; RF và bộ MFCC là các đối chiếu được báo cáo, không chọn/hậu chỉnh theo test. Điểm số chỉ xuất hiện sau khi chạy dữ liệu thật.

In [ ]:
# Ô 1 · CẤU HÌNH VÀ THƯ VIỆN
import os,sys,subprocess,importlib.util,importlib,glob,json,hashlib,time,shutil,math
from pathlib import Path
# Giới hạn số luồng để không vượt CPU Kaggle.
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[_v]="1"
NGUON=Path("/kaggle/input/notebooks/duyninhus/phuhoi-sinhbu-tudong/pa2")
RA=Path("/kaggle/working/buoc2_ketqua")
RA.mkdir(parents=True,exist_ok=True)
SEED=20261103
SR_DICH=16000
SO_WORKERS=2
KICH_THUOC_LO=128
CVM=3
LOAI_CAU={"S0104","S0116","S0117","S0245","S0248"}
GIONG_LOI="u_linh"
BO_SINH=["vieneu","omnivoice","omdiom"]
GIONG=["ke","u_linh","giong_nu_tre","giong_ong_gia","giong_trai_tre","giong_tre_em","tran_thy"]

def dam_bao(pkg,mod):
    if importlib.util.find_spec(mod) is not None:
        print("✓",pkg,"đã có");return
    print("Đang cài",pkg,"(cần Internet)...")
    p=subprocess.run([sys.executable,"-m","pip","install","-q",pkg],capture_output=True,text=True)
    importlib.invalidate_caches()
    if p.returncode!=0 or importlib.util.find_spec(mod) is None:
        raise RuntimeError("Không cài được "+pkg+": "+(p.stderr or p.stdout)[-1200:])
for a,b in [("numpy","numpy"),("pandas","pandas"),("scipy","scipy"),("soundfile","soundfile"),("scikit-learn","sklearn"),("matplotlib","matplotlib"),("joblib","joblib")]:dam_bao(a,b)
import numpy as np, pandas as pd, soundfile as sf, scipy, sklearn, joblib, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.signal import resample_poly, stft
from scipy.fft import dct
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score,precision_recall_fscore_support,confusion_matrix,classification_report
print("Nguồn:",NGUON,"\nKết quả:",RA,"\nCPU workers:",SO_WORKERS,"· mẫu:",SR_DICH,"Hz")


In [ ]:
# Ô 2 · ĐỌC MANIFEST VÀ LỌC GHÉP CẶP
mf_path=NGUON/"manifest.csv";cau_path=NGUON/"bang_cau_chuan.csv"
if not mf_path.is_file() or not cau_path.is_file():
    raise FileNotFoundError(f"Không thấy manifest.csv hoặc bang_cau_chuan.csv tại {NGUON}. Hãy Add Input đúng notebook output phục hồi.")
mf=pd.read_csv(mf_path);cau=pd.read_csv(cau_path)
cols={"bo_sinh","giong","cau_id","sr","giay","so_am_tiet"}
if not cols.issubset(mf.columns) or not {"cau_id","tap","noi_dung"}.issubset(cau.columns):
    raise ValueError("Thiếu cột trong manifest hoặc bảng câu.")
if len(mf)!=6300 or mf.duplicated(["bo_sinh","giong","cau_id"]).any():
    raise ValueError(f"Cần manifest gốc 6300 tổ hợp duy nhất; nhận được {len(mf)} dòng.")
if set(mf.bo_sinh)!=set(BO_SINH) or set(mf.giong)!=set(GIONG) or mf.cau_id.nunique()!=300:
    raise ValueError("Nhãn bộ sinh/giọng hoặc số câu không khớp dự kiến.")
if cau.cau_id.duplicated().any() or set(cau.tap)!={"train","test"}:
    raise ValueError("Bảng câu không hợp lệ hoặc có ID trùng.")
mask=(mf.giong==GIONG_LOI)&mf.cau_id.isin(LOAI_CAU)
removed=mf.loc[mask].copy()
if len(removed)!=15 or set(removed.bo_sinh)!=set(BO_SINH):
    raise ValueError("Không tìm đủ 15 clip cần loại. Dừng để tránh lệch thiết kế ghép cặp.")
d=mf.loc[~mask].copy().merge(cau[["cau_id","tap","noi_dung"]],on="cau_id",how="left",validate="many_to_one")
if d.tap.isna().any() or len(d)!=6285:raise ValueError("Không ghép được câu/tập hoặc sai số clip sau lọc.")
d["duong_dan_input"]=[str(NGUON/"audio"/b/g/(c+".flac")) for b,g,c in zip(d.bo_sinh,d.giong,d.cau_id)]
d["id_clip"]=d.bo_sinh+"/"+d.giong+"/"+d.cau_id
thieu=[p for p in d.duong_dan_input if not os.path.isfile(p)]
if thieu:raise FileNotFoundError(f"Thiếu {len(thieu)} audio; ví dụ: {thieu[:3]}. Không train trên dữ liệu thiếu.")
if set(d[d.tap=="train"].cau_id)&set(d[d.tap=="test"].cau_id):raise ValueError("Rò rỉ ID câu train/test.")
if d.tap.value_counts().to_dict()!={"train":5031,"test":1254}:
    raise ValueError("Số clip train/test không khớp thiết kế 5031/1254.")
removed.to_csv(RA/"15_clip_loai.csv",index=False,encoding="utf-8-sig")
d.to_csv(RA/"manifest_su_dung.csv",index=False,encoding="utf-8-sig")
print("Đã loại đồng bộ 15 clip (không xóa file nguồn).")
print("\nSố clip theo bộ sinh và tập:")
print(pd.crosstab(d.bo_sinh,d.tap,margins=True).to_string())
print("\nSố clip theo giọng:",d.giong.value_counts().to_dict())
print("\nKhông trùng ID câu train/test. Lưu ý: câu gần mẫu chữ có thể vẫn trùng thông tin.")


In [ ]:
# Ô 3 · MÃ TRÍCH ĐẶC TRƯNG — 16 kHz, CÙNG QUY TRÌNH CHO 3 BỘ SINH
# Tự triển khai STFT + thang mel + DCT bằng numpy/scipy; không cần TTS hay GPU.
def _mel_filters(sr=16000,nfft=512,nmels=40):
    # Thang mel HTK và dải tam giác; tính một lần mỗi tiến trình.
    hz=np.linspace(0,sr/2,nfft//2+1)
    mel=lambda h:2595*np.log10(1+h/700)
    inv=lambda m:700*(10**(m/2595)-1)
    edges=inv(np.linspace(mel(0),mel(sr/2),nmels+2))
    flt=np.zeros((nmels,len(hz)),dtype=np.float64)
    for k in range(nmels):
        left=(hz-edges[k])/(edges[k+1]-edges[k]+1e-12)
        right=(edges[k+2]-hz)/(edges[k+2]-edges[k+1]+1e-12)
        flt[k]=np.maximum(0,np.minimum(left,right))
    return flt
MEL=_mel_filters()

def dac_trung_mot(row):
    from math import gcd
    try:
        y,sr=sf.read(row["duong_dan_input"],dtype="float32",always_2d=True)
        if sr<=0 or not y.size:raise ValueError("âm rỗng hoặc sample rate không hợp lệ")
        if int(sr)!=int(row["sr"]):raise ValueError(f"sample rate thực {sr} khác manifest {row['sr']}")
        y=y.mean(axis=1).astype(np.float64)
        if not np.isfinite(y).all():raise ValueError("âm có NaN/Inf")
        if sr!=SR_DICH:
            g=gcd(int(sr),SR_DICH);y=resample_poly(y,SR_DICH//g,int(sr)//g)
        if len(y)<int(0.2*SR_DICH):raise ValueError("clip dưới 0,2s")
        raw_rms=float(np.sqrt(np.mean(y*y)))
        if raw_rms<1e-8:raise ValueError("âm hoàn toàn im lặng")
        # Chuẩn hóa RMS clip để giảm sự phụ thuộc vào âm lượng đầu ra.
        y=y/raw_rms
        _,_,Z=stft(y,fs=SR_DICH,window="hann",nperseg=400,noverlap=240,nfft=512,boundary="zeros",padded=True)
        power=np.abs(Z)**2+1e-20
        logmel=np.log(MEL@power+1e-12)
        m=dct(logmel,type=2,axis=0,norm="ortho")[:13]
        dm=np.gradient(m,axis=1) if m.shape[1]>1 else np.zeros_like(m)
        f={}
        for name,A in [("mfcc",m),("delta",dm)]:
            for i in range(13):
                f[f"F1_{name}{i:02d}_mean"]=float(np.mean(A[i]))
                f[f"F1_{name}{i:02d}_std"]=float(np.std(A[i]))
        freq=np.linspace(0,SR_DICH/2,power.shape[0]);total=power.sum(axis=0)+1e-20
        centroid=(freq[:,None]*power).sum(axis=0)/total
        bandwidth=np.sqrt(((freq[:,None]-centroid)**2*power).sum(axis=0)/total)
        roll=freq[(np.cumsum(power,axis=0)>=.95*total).argmax(axis=0)]
        flat=np.exp(np.log(power).mean(axis=0))/(power.mean(axis=0)+1e-20)
        for name,v in [("centroid_hz",centroid),("bandwidth_hz",bandwidth),("rolloff95_hz",roll),("flatness",flat)]:
            f[f"F2_{name}_mean"]=float(np.mean(v));f[f"F2_{name}_std"]=float(np.std(v))
        for a,b in [(0,500),(500,1000),(1000,2000),(2000,4000),(4000,8001)]:
            ratio=power[(freq>=a)&(freq<b)].sum(axis=0)/total
            f[f"F2_band_{a}_{b}_mean"]=float(np.mean(ratio))
            f[f"F2_band_{a}_{b}_std"]=float(np.std(ratio))
        n=400;hop=160
        yp=np.pad(y,(n//2,n//2));fr=np.lib.stride_tricks.sliding_window_view(yp,n)[::hop]
        er=np.sqrt(np.mean(fr*fr,axis=1));zc=np.mean((fr[:,:-1]*fr[:,1:])<0,axis=1)
        threshold=max(float(er.max())*0.1,1e-12)
        f.update({"F3_duration_s":float(len(y)/SR_DICH),"F3_silence_fraction":float(np.mean(er<threshold)),
                  "F3_frame_rms_std":float(np.std(er)),"F3_zcr_mean":float(np.mean(zc)),"F3_zcr_std":float(np.std(zc))})
        if not all(np.isfinite(v) for v in f.values()):raise ValueError("Đặc trưng có NaN/Inf")
        return {"id_clip":row["id_clip"],**f},None
    except Exception as e:
        return None,{"id_clip":row["id_clip"],"duong_dan":row["duong_dan_input"],"loi":f"{type(e).__name__}: {str(e)[:300]}"}
print("F1: MFCC + delta (mean/std); F2: phổ; F3: thời lượng và nhịp. Không trích F0.")


In [ ]:
# Ô 4 · TRÍCH ĐẶC TRƯNG THEO LÔ, LƯU ĐỆM TẠI WORKING
# Kết quả cache chỉ được dùng lại khi manifest, nguồn, cấu hình và phiên bản mã khớp.
cache=RA/"dac_trung_cache.csv";meta_cache=RA/"dac_trung_cache_meta.json"
sign=hashlib.sha256(mf_path.read_bytes()+cau_path.read_bytes()+(str(NGUON)+"|v2_f1f2f3_16k_rms_norm").encode()).hexdigest()
if cache.exists() and meta_cache.exists():
    info=json.loads(meta_cache.read_text(encoding="utf-8"))
    if info.get("signature")!=sign:raise RuntimeError("Cache khác nguồn/cấu hình; dừng để tránh trộn dữ liệu.")
    feat=pd.read_csv(cache)
    if feat.id_clip.duplicated().any() or not set(feat.id_clip).issubset(set(d.id_clip)):raise ValueError("Cache chứa ID trùng hoặc ngoài dữ liệu.")
    print("Dùng lại",len(feat),"dòng đặc trưng từ cache.")
else:
    if cache.exists() or meta_cache.exists():raise RuntimeError("Cache không đầy đủ metadata; không tự xóa hay trộn.")
    feat=pd.DataFrame(columns=["id_clip"]);meta_cache.write_text(json.dumps({"signature":sign,"nguon":str(NGUON)},ensure_ascii=False),encoding="utf-8")
co=set(feat.id_clip) if not feat.empty else set()
con=d[~d.id_clip.isin(co)].to_dict("records")
print(f"Cần trích {len(con)} / {len(d)} clip; mỗi lô {KICH_THUOC_LO}.")
t0=time.time();errs=[]
for start in range(0,len(con),KICH_THUOC_LO):
    batch=con[start:start+KICH_THUOC_LO]
    got=Parallel(n_jobs=SO_WORKERS,prefer="threads")(delayed(dac_trung_mot)(x) for x in batch)
    ok=[a for a,b in got if a is not None];errs.extend(b for a,b in got if b is not None)
    if ok:feat=pd.concat([feat,pd.DataFrame(ok)],ignore_index=True)
    temp=cache.with_suffix(".tmp")
    feat.to_csv(temp,index=False,encoding="utf-8-sig");os.replace(temp,cache)
    print(f"  {min(start+len(batch),len(con))}/{len(con)} · lỗi {len(errs)} · {(time.time()-t0)/60:.1f} phút",flush=True)
if errs:
    pd.DataFrame(errs).to_csv(RA/"loi_trich_dac_trung.csv",index=False,encoding="utf-8-sig")
    raise RuntimeError(f"Có {len(errs)} clip lỗi đọc/trích; đã lưu loi_trich_dac_trung.csv. Dừng, không bỏ clip âm thầm.")
if feat.empty or len(feat)!=len(d):raise RuntimeError("Số dòng đặc trưng chưa đủ; xem cache và chạy lại khi dữ liệu sẵn sàng.")
d=d.drop(columns=[c for c in d.columns if c.startswith(("F1_","F2_","F3_"))],errors="ignore").merge(feat,on="id_clip",how="inner",validate="one_to_one")
f1=sorted([c for c in feat.columns if c.startswith("F1_")])
f2=sorted([c for c in feat.columns if c.startswith("F2_")])
f3=sorted([c for c in feat.columns if c.startswith("F3_")])
ALL=f1+f2+f3
if not f1 or not f2 or not f3 or not np.isfinite(d[ALL].to_numpy(dtype=float)).all():raise ValueError("Thiếu đặc trưng hoặc có NaN/Inf.")
print(f"✅ {len(d)} clip · F1={len(f1)}, F2={len(f2)}, F3={len(f3)} đặc trưng")


In [ ]:
# Ô 5 · HUẤN LUYỆN / CV NHÓM THEO CÂU / TEST ĐỘC LẬP
# Hai nhóm đặc trưng: MFCC (F1) và đầy đủ (F1+F2+F3).
# Mô hình chính định trước: SVM + đầy đủ. CV nhóm câu chỉ chạy trên train.
sets={"F1_MFCC":f1,"F1_F2_F3":ALL}
models={"SVM":lambda:Pipeline([("scale",StandardScaler()),("model",SVC(C=1.0,kernel="rbf",class_weight="balanced"))]),
        "RandomForest":lambda:RandomForestClassifier(n_estimators=150,min_samples_leaf=2,class_weight="balanced",n_jobs=SO_WORKERS,random_state=SEED)}
tasks={"T_B_bo_sinh":("bo_sinh",["vieneu","omnivoice","omdiom"]),"T_C_giong":("giong",GIONG)}
train=d[d.tap=="train"].copy();test=d[d.tap=="test"].copy()
folds=list(GroupKFold(n_splits=CVM).split(train,groups=train.cau_id))
for k,(tr,va) in enumerate(folds,1):
    if set(train.iloc[tr].cau_id)&set(train.iloc[va].cau_id):raise RuntimeError("Lỗi tách nhóm câu CV.")
    for _,(target,labels) in tasks.items():
        if set(train.iloc[tr][target])!=set(labels) or set(train.iloc[va][target])!=set(labels):raise RuntimeError(f"Fold {k} thiếu nhãn {target}.")
ketqua=[]; predicts=[];chi_tiet={}; modeldir=RA/"mo_hinh";modeldir.mkdir(exist_ok=True)
for task,(target,labels) in tasks.items():
    for featureset,columns in sets.items():
        Xtr=train[columns].to_numpy(dtype=np.float64);Xte=test[columns].to_numpy(dtype=np.float64)
        ytr=train[target].to_numpy();yte=test[target].to_numpy()
        for modelname,make in models.items():
            print(f"\n▶ {task} · {featureset} · {modelname}",flush=True);start=time.time();scores=[]
            for k,(tr,va) in enumerate(folds,1):
                model=make();model.fit(Xtr[tr],ytr[tr]);pred=model.predict(Xtr[va])
                _,_,f,_=precision_recall_fscore_support(ytr[va],pred,labels=labels,average="macro",zero_division=0)
                scores.append(float(f));print(f"  CV {k}/{CVM}: macro-F1={f:.4f}",flush=True)
            model=make();model.fit(Xtr,ytr);pred=model.predict(Xte)
            acc=accuracy_score(yte,pred);pr,re,f,_=precision_recall_fscore_support(yte,pred,labels=labels,average="macro",zero_division=0)
            cm=confusion_matrix(yte,pred,labels=labels)
            name=task+"__"+featureset+"__"+modelname
            # Lưu model cùng thứ tự features và lớp; scaler đã nằm trong SVM Pipeline.
            joblib.dump({"model":model,"feature_names":columns,"labels":labels,"sr":SR_DICH,"feature_version":"v2_f1f2f3_16k_rms_norm","note":"Can cung ma trich dac trung trong notebook de suy luan."},modeldir/(name+".joblib"))
            row={"bai_toan":task,"dac_trung":featureset,"mo_hinh":modelname,"cv_macro_f1_tb":float(np.mean(scores)),"cv_macro_f1_do_lech_chuan":float(np.std(scores,ddof=0)),"test_accuracy":float(acc),"test_precision_macro":float(pr),"test_recall_macro":float(re),"test_f1_macro":float(f),"so_train":len(train),"so_test":len(test),"giay":round(time.time()-start,2)}
            ketqua.append(row)
            detail=classification_report(yte,pred,labels=labels,output_dict=True,zero_division=0)
            chi_tiet[name]={"labels":labels,"confusion_matrix":cm.tolist(),"classification_report":detail,"cv_f1":scores}
            dd=test[["id_clip","cau_id","giong","bo_sinh"]].copy();dd["bai_toan"]=task;dd["dac_trung"]=featureset;dd["mo_hinh"]=modelname;dd["nhan_dung"]=yte;dd["du_doan"]=pred
            predicts.append(dd)
            print(f"  TEST accuracy={acc:.4f} · macro-F1={f:.4f} · {time.time()-start:.1f}s",flush=True)
            pd.DataFrame(ketqua).to_csv(RA/"ket_qua.csv",index=False,encoding="utf-8-sig")
            with open(RA/"chi_tiet_phan_loai.json","w",encoding="utf-8") as out:json.dump(chi_tiet,out,ensure_ascii=False,indent=2)
pd.concat(predicts,ignore_index=True).to_csv(RA/"du_doan_test.csv",index=False,encoding="utf-8-sig")
print("\n✅ Huấn luyện và đánh giá xong. Không chọn mô hình dựa vào điểm test.")


In [ ]:
# Ô 6 · BÁO CÁO HTML TỰ CHỨA — KẾT QUẢ THẬT VÀ CHÚ THÍCH
import io,base64,html
plt.rcParams.update({"figure.facecolor":"white","axes.spines.top":False,"axes.spines.right":False,"font.size":10.5,"axes.titlesize":13,"axes.titleweight":"bold","figure.autolayout":True})
figures=[]
def savefig(title,explain,fig):
    b=io.BytesIO();fig.savefig(b,format="png",dpi=145,bbox_inches="tight",facecolor="white");plt.close(fig)
    figures.append((title,explain,base64.b64encode(b.getvalue()).decode()))

fig,ax=plt.subplots(figsize=(9.5,3.3));ct=pd.crosstab(d.giong,d.bo_sinh).reindex(index=GIONG,columns=BO_SINH)
im=ax.imshow(ct.values,cmap="YlGnBu",aspect="auto");ax.set_xticks(range(3),BO_SINH);ax.set_yticks(range(7),GIONG)
lo,hi=float(ct.values.min()),float(ct.values.max())
for i in range(7):
    for j in range(3):
        frac=(ct.values[i,j]-lo)/(hi-lo) if hi>lo else 0
        ax.text(j,i,int(ct.values[i,j]),ha="center",va="center",weight="bold",color="white" if frac>.55 else "#152230")
ax.set_title("Số clip theo nhãn sau lọc");ax.grid(False)
if hi>lo:fig.colorbar(im,ax=ax,label="clip",shrink=.8)
savefig("Phân bố dữ liệu","Mỗi giọng có số lượng bằng nhau giữa ba bộ sinh; u_linh giảm 5 câu cho cả ba.",fig)

fig,ax=plt.subplots(figsize=(8,3.5));ct2=pd.crosstab(d.bo_sinh,d.tap).reindex(index=BO_SINH,columns=["train","test"])
ct2.plot(kind="bar",ax=ax,color=["#237d9a","#ef9354"],edgecolor="white",rot=0);ax.set_ylabel("số clip");ax.set_title("Phân chia theo câu");ax.legend(frameon=False)
savefig("Train và test","Đã tách theo ID câu. Những câu có cùng mẫu chữ vẫn có thể làm đánh giá lạc quan.",fig)

fig,ax=plt.subplots(figsize=(9,3.6)); vals=[d[d.bo_sinh==b].giay.values for b in BO_SINH]
bp=ax.boxplot(vals,patch_artist=True,widths=.55,showfliers=False)
for p,c in zip(bp["boxes"],["#2e86ab","#e4572e","#17a398"]):p.set_facecolor(c);p.set_alpha(.75)
ax.set_xticks(range(1,4),BO_SINH);ax.set_ylabel("giây");ax.set_title("Thời lượng sau khi lọc");ax.grid(axis="y",alpha=.2)
savefig("Phân bố thời lượng","Thời lượng có thể là lối tắt phân loại; độ dài không chứng minh câu được đọc đúng.",fig)

results=pd.DataFrame(ketqua)
for task,(target,labels) in tasks.items():
    sub=results[results.bai_toan==task]
    fig,ax=plt.subplots(figsize=(9,3.6));x=np.arange(len(sub));colors=["#2e86ab" if m=="SVM" else "#ef9354" for m in sub.mo_hinh]
    ax.bar(x,sub.test_f1_macro,color=colors,width=.6)
    ax.set_xticks(x,[f"{a}\n{b}" for a,b in zip(sub.dac_trung,sub.mo_hinh)]);ax.set_ylim(0,1.08);ax.set_ylabel("macro-F1 trên test");ax.set_title(task+" · so sánh mô tả")
    for i,v in enumerate(sub.test_f1_macro):ax.text(i,v+.025,f"{v:.3f}",ha="center",weight="bold")
    ax.grid(axis="y",alpha=.2)
    savefig(task+" — macro-F1", "Điểm test chỉ để báo cáo; mô hình chính SVM với đầy đủ đặc trưng đã được định trước. Không chọn mô hình theo test.",fig)
    # Ma trận của mô hình chính SVM + đầy đủ; cả 8 ma trận có trong JSON.
    key=task+"__F1_F2_F3__SVM";cm=np.array(chi_tiet[key]["confusion_matrix"]);lab=chi_tiet[key]["labels"]
    norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    fig,ax=plt.subplots(figsize=(max(6,len(lab)*1.08),max(4,len(lab)*.72)));im=ax.imshow(norm,cmap="Blues",vmin=0,vmax=1)
    ax.set_xticks(range(len(lab)),lab,rotation=30,ha="right");ax.set_yticks(range(len(lab)),lab);ax.set_xlabel("Dự đoán");ax.set_ylabel("Nhãn đúng");ax.set_title(task+" · SVM đầy đủ · test")
    for i in range(len(lab)):
        for j in range(len(lab)):
            ax.text(j,i,f"{cm[i,j]}\n({norm[i,j]*100:.0f}%)",ha="center",va="center",fontsize=8.5,color="white" if norm[i,j]>.52 else "#17232d")
    ax.grid(False);fig.colorbar(im,ax=ax,shrink=.75,label="tỷ lệ trong hàng")
    savefig(task+" — ma trận nhầm lẫn", "Mỗi hàng là nhãn thật; số trong ô là số clip và tỷ lệ theo hàng. Đây là nhãn bộ sinh/prompt, không phải nhãn thật–giả hay người thật.",fig)

# Phân bố điểm số F1 và F3, chỉ mô tả dữ liệu chứ không chứng minh nguyên nhân.
fig,ax=plt.subplots(figsize=(9,3.6))
for b,col in zip(BO_SINH,["#2e86ab","#e4572e","#17a398"]):
    v=d[d.bo_sinh==b]["F2_rolloff95_hz_mean"]
    ax.hist(v,bins=35,alpha=.44,label=b,color=col,density=True)
ax.set_xlabel("Rolloff 95% năng lượng trung bình (Hz)");ax.set_ylabel("mật độ");ax.set_title("Phân bố rolloff tại 16 kHz");ax.legend(frameon=False)
savefig("Đặc trưng phổ sau resample", "Dùng phổ công suất, hiển thị theo Hz. Resample không đảm bảo loại bỏ mọi khác biệt băng thông/codec.",fig)

# Bảng kết quả và bản phân loại theo lớp của mô hình chính.
show=results.copy();show.columns=["bài toán","đặc trưng","mô hình","CV macro-F1 TB","CV độ lệch chuẩn","test accuracy","test precision macro","test recall macro","test macro-F1","train","test","giây"]
table=show.to_html(index=False,border=0,classes="table",float_format=lambda x:f"{x:.3f}",escape=True)
sections=[]
for title,explain,img in figures:
    sections.append("<figure><h3>"+html.escape(title)+"</h3><img src='data:image/png;base64,"+img+"'><figcaption>"+html.escape(explain)+"</figcaption></figure>")
class_tables=[]
for task in tasks:
    key=task+"__F1_F2_F3__SVM";report=chi_tiet[key]["classification_report"];lab=chi_tiet[key]["labels"]
    t=pd.DataFrame([{"nhãn":label,"precision":report[label]["precision"],"recall":report[label]["recall"],"f1":report[label]["f1-score"],"số mẫu":int(report[label]["support"])} for label in lab])
    class_tables.append("<h3>"+html.escape(task)+" · SVM đầy đủ</h3>"+t.to_html(index=False,border=0,classes="table",float_format=lambda x:f"{x:.3f}",escape=True))
style="""body{margin:0;background:#f4f7fa;color:#152638;font:16px/1.55 system-ui,-apple-system,Segoe UI,sans-serif}header{background:linear-gradient(120deg,#12374b,#21819a);color:white;padding:38px 24px}header div,main{max-width:1120px;margin:auto}main{padding:24px 20px 70px}h1{margin:0}h2{margin-top:40px;border-bottom:2px solid #dce6ed;color:#155b75}figure,.box{background:white;border:1px solid #dce5ed;border-radius:12px;padding:18px;margin:18px 0;overflow:auto}figure img{max-width:100%;display:block;margin:auto}figcaption{border-left:3px solid #2191a7;padding-left:12px;color:#506271;font-size:14px}table.table{border-collapse:collapse;width:100%;font-size:13px;white-space:nowrap}table.table th,table.table td{padding:8px;border-bottom:1px solid #e2eaf0;text-align:left}table.table th{background:#eaf3f7}.warn{background:#fff8e9;border-left:4px solid #e9a740;padding:15px}.tag{display:inline-block;padding:4px 10px;background:#ffffff2b;border-radius:20px;margin:10px 5px 0 0} @media print{body{background:white}figure{break-inside:avoid}}"""
summary=f"{len(d):,} clip · {len(train):,} train · {len(test):,} test · {SR_DICH:,} Hz · {len(ALL)} đặc trưng"
page=["<!doctype html><html lang='vi'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>Bước 2 — Nhận dạng</title><style>",style,"</style></head><body><header><div><h1>Bước 2 · Nhận dạng bộ sinh & giọng</h1><p>Nhóm 05 · Báo cáo số liệu từ lần chạy thực tế</p><span class='tag'>",html.escape(summary),"</span></div></header><main>"]
page.append("<div class='warn'><b>Giới hạn:</b> đã loại 15 clip của 5 câu lỗi theo cặp; các clip còn lại chưa được xác minh toàn bộ bằng tai. Tần số mẫu khác nhau ban đầu; resample không xóa hết dấu vết xử lý. Train/test tách theo ID câu nhưng các câu gần mẫu có thể vẫn tương tự. T-C nhận dạng 7 nhãn prompt trong bộ âm tổng hợp, không chứng minh nhận dạng người thật mới. Đây không phải bài toán thật/giả.</div>")
page.append("<h2>Thiết kế và kết quả</h2><div class='box'><p>CV "+str(CVM)+" fold nhóm theo ID câu trên train; test giữ riêng. SVM đầy đủ là mô hình chính định trước; RF và F1 là đối chiếu mô tả. Không điều chỉnh/chọn mô hình theo test.</p>"+table+"</div>")
page.append("<h2>Hình trực quan</h2>"+"".join(sections));page.append("<h2>Chi tiết theo lớp trên test</h2><div class='box'>"+"".join(class_tables)+"</div>")
page.append("<p>Đặc trưng: F1 gồm 13 MFCC + delta (mean/std); F2 là centroid, bandwidth, rolloff 95%, flatness, tỷ lệ năng lượng 5 dải; F3 là thời lượng, khoảng năng lượng thấp, RMS khung và zero-crossing. Chuẩn hóa RMS theo clip trước trích xuất; tần số 16 kHz. Mã trích đặc trưng nằm trong notebook.</p>")
page.append("<p>Output: ket_qua.csv · chi_tiet_phan_loai.json · du_doan_test.csv · dac_trung_cache.csv · manifest_su_dung.csv · mo_hinh/.</p></main></body></html>")
report=RA/"BAO_CAO_BUOC2.html";report.write_text("".join(page),encoding="utf-8")
print(f"✅ {report} · {len(figures)} hình · {report.stat().st_size/1024**2:.2f} MiB")


In [ ]:
(RA/"bo_trich_dac_trung.py").write_text('import numpy as np\nimport soundfile as sf\nfrom scipy.signal import resample_poly, stft\nfrom scipy.fft import dct\nSR_DICH=16000\n# Ô 3 · MÃ TRÍCH ĐẶC TRƯNG — 16 kHz, CÙNG QUY TRÌNH CHO 3 BỘ SINH\n# Tự triển khai STFT + thang mel + DCT bằng numpy/scipy; không cần TTS hay GPU.\ndef _mel_filters(sr=16000,nfft=512,nmels=40):\n    # Thang mel HTK và dải tam giác; tính một lần mỗi tiến trình.\n    hz=np.linspace(0,sr/2,nfft//2+1)\n    mel=lambda h:2595*np.log10(1+h/700)\n    inv=lambda m:700*(10**(m/2595)-1)\n    edges=inv(np.linspace(mel(0),mel(sr/2),nmels+2))\n    flt=np.zeros((nmels,len(hz)),dtype=np.float64)\n    for k in range(nmels):\n        left=(hz-edges[k])/(edges[k+1]-edges[k]+1e-12)\n        right=(edges[k+2]-hz)/(edges[k+2]-edges[k+1]+1e-12)\n        flt[k]=np.maximum(0,np.minimum(left,right))\n    return flt\nMEL=_mel_filters()\n\ndef dac_trung_mot(row):\n    from math import gcd\n    try:\n        y,sr=sf.read(row["duong_dan_input"],dtype="float32",always_2d=True)\n        if sr<=0 or not y.size:raise ValueError("âm rỗng hoặc sample rate không hợp lệ")\n        if int(sr)!=int(row["sr"]):raise ValueError(f"sample rate thực {sr} khác manifest {row[\'sr\']}")\n        y=y.mean(axis=1).astype(np.float64)\n        if not np.isfinite(y).all():raise ValueError("âm có NaN/Inf")\n        if sr!=SR_DICH:\n            g=gcd(int(sr),SR_DICH);y=resample_poly(y,SR_DICH//g,int(sr)//g)\n        if len(y)<int(0.2*SR_DICH):raise ValueError("clip dưới 0,2s")\n        raw_rms=float(np.sqrt(np.mean(y*y)))\n        if raw_rms<1e-8:raise ValueError("âm hoàn toàn im lặng")\n        # Chuẩn hóa RMS clip để giảm sự phụ thuộc vào âm lượng đầu ra.\n        y=y/raw_rms\n        _,_,Z=stft(y,fs=SR_DICH,window="hann",nperseg=400,noverlap=240,nfft=512,boundary="zeros",padded=True)\n        power=np.abs(Z)**2+1e-20\n        logmel=np.log(MEL@power+1e-12)\n        m=dct(logmel,type=2,axis=0,norm="ortho")[:13]\n        dm=np.gradient(m,axis=1) if m.shape[1]>1 else np.zeros_like(m)\n        f={}\n        for name,A in [("mfcc",m),("delta",dm)]:\n            for i in range(13):\n                f[f"F1_{name}{i:02d}_mean"]=float(np.mean(A[i]))\n                f[f"F1_{name}{i:02d}_std"]=float(np.std(A[i]))\n        freq=np.linspace(0,SR_DICH/2,power.shape[0]);total=power.sum(axis=0)+1e-20\n        centroid=(freq[:,None]*power).sum(axis=0)/total\n        bandwidth=np.sqrt(((freq[:,None]-centroid)**2*power).sum(axis=0)/total)\n        roll=freq[(np.cumsum(power,axis=0)>=.95*total).argmax(axis=0)]\n        flat=np.exp(np.log(power).mean(axis=0))/(power.mean(axis=0)+1e-20)\n        for name,v in [("centroid_hz",centroid),("bandwidth_hz",bandwidth),("rolloff95_hz",roll),("flatness",flat)]:\n            f[f"F2_{name}_mean"]=float(np.mean(v));f[f"F2_{name}_std"]=float(np.std(v))\n        for a,b in [(0,500),(500,1000),(1000,2000),(2000,4000),(4000,8001)]:\n            ratio=power[(freq>=a)&(freq<b)].sum(axis=0)/total\n            f[f"F2_band_{a}_{b}_mean"]=float(np.mean(ratio))\n            f[f"F2_band_{a}_{b}_std"]=float(np.std(ratio))\n        n=400;hop=160\n        yp=np.pad(y,(n//2,n//2));fr=np.lib.stride_tricks.sliding_window_view(yp,n)[::hop]\n        er=np.sqrt(np.mean(fr*fr,axis=1));zc=np.mean((fr[:,:-1]*fr[:,1:])<0,axis=1)\n        threshold=max(float(er.max())*0.1,1e-12)\n        f.update({"F3_duration_s":float(len(y)/SR_DICH),"F3_silence_fraction":float(np.mean(er<threshold)),\n                  "F3_frame_rms_std":float(np.std(er)),"F3_zcr_mean":float(np.mean(zc)),"F3_zcr_std":float(np.std(zc))})\n        if not all(np.isfinite(v) for v in f.values()):raise ValueError("Đặc trưng có NaN/Inf")\n        return {"id_clip":row["id_clip"],**f},None\n    except Exception as e:\n        return None,{"id_clip":row["id_clip"],"duong_dan":row["duong_dan_input"],"loi":f"{type(e).__name__}: {str(e)[:300]}"}\nprint("F1: MFCC + delta (mean/std); F2: phổ; F3: thời lượng và nhịp. Không trích F0.")\n',encoding="utf-8")
(RA/"HUONG_DAN_SU_DUNG_MO_HINH.md").write_text('# Dùng lại mô hình đã huấn luyện\n\nChạy ở thư mục đã giải nén, cài numpy, scipy, soundfile, scikit-learn và joblib; nên dùng cùng phiên bản ghi trong thong_tin_thi_nghiem.json. Tệp .joblib chỉ nên mở nếu bạn tin tưởng nguồn.\n\n```python\nimport numpy as np\nimport soundfile as sf\nimport joblib\nfrom bo_trich_dac_trung import dac_trung_mot\n\np = "duong_dan_toi_clip.flac"  # thay bằng tệp thực tế khi suy luận\nsr = sf.info(p).samplerate\nrow = {"id_clip": "clip_moi", "duong_dan_input": p, "sr": sr}\nf, loi = dac_trung_mot(row)\nif loi:\n    raise RuntimeError(loi)\nb = joblib.load("mo_hinh/T_B_bo_sinh__F1_F2_F3__SVM.joblib")\nX = np.array([[f[k] for k in b["feature_names"]]], dtype=float)\nprint("Nhãn dự đoán:", b["model"].predict(X)[0])\n```\n\nBộ phân loại T-B dự đoán một trong ba bộ sinh đã huấn luyện; T-C dự đoán một trong bảy nhãn prompt đã có. Không dùng các mô hình này để khẳng định thật/giả hoặc nhận dạng người nói mới.\n',encoding="utf-8")
# Ô 7 · LƯU MÔ TẢ VÀ ĐÓNG GÓI
meta={"nguon_input":str(NGUON),"manifest_sha256":hashlib.sha256(mf_path.read_bytes()).hexdigest(),
      "bang_cau_sha256":hashlib.sha256(cau_path.read_bytes()).hexdigest(),"seed":SEED,"sr_dich":SR_DICH,
      "loai":{"giong":GIONG_LOI,"cau":sorted(LOAI_CAU),"bo_sinh":BO_SINH,"so_clip":15},
      "so_train":len(train),"so_test":len(test),"cv":"GroupKFold 3, theo cau_id tren train",
      "feature_version":"v2_f1f2f3_16k_rms_norm","feature_names":ALL,
      "versions":{"python":sys.version,"numpy":np.__version__,"pandas":pd.__version__,"scipy":scipy.__version__,"soundfile":sf.__version__,"sklearn":sklearn.__version__,"joblib":joblib.__version__,"matplotlib":matplotlib.__version__},
      "gioi_han":"T-B/T-C tren giong tong hop; khong phai T-A; split theo ID cau nhung mau cau gan nhau; dac trung co the hoc dau vet xu ly"}
(RA/"thong_tin_thi_nghiem.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
ten="/kaggle/working/KET_QUA_BUOC2_"+time.strftime("%m%d_%H%M%S")
zip_path=shutil.make_archive(ten,"zip",RA)
print("\n📄 HTML:",report)
print(f"📦 ZIP: {zip_path} · {os.path.getsize(zip_path)/1024**2:.1f} MiB")
print("Gửi BAO_CAO_BUOC2.html + ket_qua.csv + chi_tiet_phan_loai.json để đọc kết quả.")
print("Đã hoàn thành. Lưu output bằng Save Version; cache working chỉ tồn tại trong phiên nếu chưa lưu output.")
